## education

In [18]:
import pandas as pd
import re
import json

In [19]:
# import dictionary of education(json)

with open(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\education\education_pattern.json",encoding='utf-8')as fin:
    pattern_dict = json.load(fin)

In [20]:
# import data and del column 7 (meaning of data)

data_edu = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\education\1_customer_comp_education.csv" , encoding = 'utf8')

data_edu.shape

(64, 8)

In [21]:
pd.set_option('display.max_rows', None)

In [22]:
data_edu.head(5)

,subject,no,id,case,detail case,flag,education,Unnamed: 7
0,education,1,education-ENG-1,ENG,normal,good,Bachelor,ปริญญาตรี
1,education,2,education-ENG-2,ENG,normal,good,Diploma,อนุปริญญา
2,education,3,education-ENG-3,ENG,normal,good,Doctor,ปริญญาเอก
3,education,4,education-ENG-4,ENG,normal,good,High_school,มัธยมศึกษาตอนปลาย
4,education,5,education-ENG-5,ENG,normal,good,Jr_high_school,มัธยมศึกษาตอนต้น


In [23]:
data_edu.tail(5)

,subject,no,id,case,detail case,flag,education,Unnamed: 7
59,education,60,education-THA-60,THA,spell,bad,อืน ๆ,NaN
60,education,61,education-THA-61,THA,spell,bad,อืน,NaN
61,education,62,education-THA-62,THA,spell,bad,ประถม,NaN
62,education,63,education-THA-63,THA,spell,bad,ป.,NaN
63,education,64,education-THA-64,THA,spell,bad,ประทม,NaN


In [24]:
# make the data to lower case
# remove . and white space
def standard_transform(text_input):
    text_input = text_input.lower().replace('.' , '').replace(' ' , '')
    return text_input



# transform data
data_edu['education_transform'] = data_edu['education'].apply(lambda row:standard_transform(row))

In [25]:
# choose only english letters
def take_eng(input_edu) :
    try :
        result = re.search(r'[a-zA-Z0-9_ ]+' , input_edu).group()    
    except AttributeError :
        return None
    return result


# create column for english letters
data_edu['df_eng'] = data_edu['education_transform'].apply(lambda education_transform : take_eng(education_transform))

In [26]:
# choose only thai letters
def take_thai(input_edu) :
    try :
        result = re.search(r'[ก-๛ 0-9]+' , input_edu).group()
    except AttributeError :
        return None
    return result



# create column for thai letters
data_edu['df_thai'] = data_edu['education_transform'].apply(lambda education_transform : take_thai(education_transform))

In [27]:
# create data frame for case = eng and case = thai
edu_eng = data_edu[data_edu['df_eng'].notnull()]
edu_thai = data_edu[data_edu['df_thai'].notnull()]

# del original data frame
del data_edu

In [28]:
# create standard format
def std_edu(text_input,lang) :
    try :
        for pattern,value in pattern_dict[lang].items():
            if re.search(pattern , text_input):
                return value
            else:
                continue
        return None
    except :
        return None
    
    
    
# create column for standard format
edu_eng['standard_education'] = edu_eng['df_eng'].apply(lambda df_eng : std_edu(df_eng , 'eng'))
edu_thai['standard_education'] = edu_thai['df_thai'].apply(lambda df_thai : std_edu(df_thai , 'tha'))

In [29]:
# combined data frame of english and thai
education = pd.concat([edu_eng , edu_thai])

In [30]:
# standard output / status
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    
    
education[['std_output' , 'status']] = education.apply(lambda row: pd.Series(result(row['education'], row['standard_education'])), axis=1)

In [31]:
selected_cols = ['id','education','standard_education' ,'std_output' ,'status']

education = education[selected_cols]

In [32]:
education.tail(5)

,id,education,standard_education,std_output,status
59,education-THA-60,อืน ๆ,Other,Other,2
60,education-THA-61,อืน,Other,Other,2
61,education-THA-62,ประถม,Primary,Primary,2
62,education-THA-63,ป.,Primary,Primary,2
63,education-THA-64,ประทม,Primary,Primary,2
